# Analisis Market Basket Dataset
Dataset: 1500 transaksi x 10 produk. Nilai kosong (NaN) = produk **tidak dibeli** pada transaksi tersebut.

In [ ]:
# Cell upload Colab (tidak digunakan pada lingkungan lokal)
pass


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('market_basket_dataset.csv')
df.head()

## 1. Total data

In [ ]:
print('Total data (baris):', df.shape[0])
print('Total sel        :', df.size)

## 2. Jumlah atribut

In [ ]:
print('Jumlah atribut:', df.shape[1])
print(list(df.columns))
df.info()

## 3. Jumlah missing value

In [ ]:
mv = df.isnull().sum()
print(mv)
print('\nTotal missing :', mv.sum())
print('Persentase    :', round(mv.sum()/df.size*100, 2), '%')
print('Baris kosong semua kolom:', df.isnull().all(axis=1).sum())

## 4. Apakah bisa ditentukan mean, median, modus?
Semua kolom bertipe **kategorikal (teks)**, dan tiap kolom hanya punya 1 nilai unik (nama produknya sendiri).
- **Mean & median: tidak bisa** (bukan angka).
- **Modus: bisa secara teknis**, tapi hasilnya selalu nama produk itu sendiri, jadi tidak informatif.

Agar mean/median/modus bermakna, data diubah ke biner (1 = dibeli, 0 = tidak).

In [ ]:
print(df.dtypes)
print('\nModus per kolom (data asli):')
print(df.mode().iloc[0])

In [ ]:
# Ubah ke biner (one-hot): 1 = dibeli, 0 = tidak dibeli
basket = df.notnull().astype(int)

stat = pd.DataFrame({
    'mean'  : basket.mean(),
    'median': basket.median(),
    'modus' : basket.mode().iloc[0]
})
stat

## 5. Menanggulangi missing value
Pada data market basket, NaN **bukan data hilang**, melainkan produk yang tidak dibeli. Jadi cara terbaik adalah **mengisi dengan 0 / mengubah ke biner**, bukan imputasi mean/modus (itu akan membuat data palsu).
Baris yang kosong semua (0 item dibeli) tidak berguna untuk analisis asosiasi, sehingga boleh dibuang.

In [ ]:
# Isi NaN -> 0, nilai produk -> 1
df_clean = df.notnull().astype(int)

# Buang transaksi kosong (tidak ada barang yang dibeli)
kosong = (df_clean.sum(axis=1) == 0)
print('Transaksi kosong dibuang:', kosong.sum())
df_clean = df_clean[~kosong].reset_index(drop=True)

print('Sisa data:', df_clean.shape)
print('Missing value sekarang:', df_clean.isnull().sum().sum())
df_clean.head()

## 6. Heatmap

In [ ]:
# (a) Heatmap missing value (kuning = kosong / tidak dibeli)
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Heatmap Missing Value')
plt.show()

In [ ]:
# (b) Heatmap korelasi antar produk
plt.figure(figsize=(9, 7))
sns.heatmap(df_clean.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Heatmap Korelasi Antar Produk')
plt.show()

## 7. Cek data record

In [ ]:
print(df.head(10))
print()
print(df.tail(5))
print()
print('Jumlah item per transaksi:')
print(df.notnull().sum(axis=1).describe())
print('\nFrekuensi pembelian tiap produk:')
print(df.notnull().sum().sort_values(ascending=False))

## 8. Cek duplikat

In [ ]:
print('Baris duplikat (selain kemunculan pertama):', df.duplicated().sum())
print('Baris unik                                :', len(df.drop_duplicates()))
print('Semua baris yang terlibat duplikasi       :', df.duplicated(keep=False).sum())
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(10)

## 9. Menanggulangi duplikat
Catatan: dengan hanya 10 produk, ada maksimal 2^10 = 1024 kombinasi keranjang. Jadi dua transaksi berbeda yang isinya sama itu **wajar** dan bisa jadi bukan duplikat sungguhan (tidak ada kolom ID transaksi). Dua pilihan:
- **Dipertahankan** bila tiap baris dianggap transaksi berbeda (disarankan untuk association rule/Apriori, karena frekuensi mempengaruhi support).
- **Dihapus** bila tugas Anda meminta membersihkan duplikat.

In [ ]:
# Opsi: hapus duplikat
df_nodup = df.drop_duplicates().reset_index(drop=True)
print('Sebelum:', df.shape[0], '| Sesudah:', df_nodup.shape[0])
print('Duplikat tersisa:', df_nodup.duplicated().sum())

## 10. Cek outlier
Outlier tidak relevan untuk kolom produk (biner). Yang bisa dicek adalah **jumlah item per transaksi**, memakai metode IQR.

In [ ]:
n_item = df_clean.sum(axis=1)

Q1, Q3 = n_item.quantile(0.25), n_item.quantile(0.75)
IQR = Q3 - Q1
batas_bawah, batas_atas = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outlier = n_item[(n_item < batas_bawah) | (n_item > batas_atas)]

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Batas bawah={batas_bawah}, batas atas={batas_atas}')
print('Jumlah outlier:', len(outlier))
print(outlier)

plt.figure(figsize=(8, 3))
sns.boxplot(x=n_item)
plt.title('Boxplot Jumlah Item per Transaksi')
plt.show()